# Database Explorer

Explore all garden databases: `plant.duckdb` and `weather.duckdb`

## plant.duckdb

In [24]:
import duckdb
import pandas as pd
from pathlib import Path

plant_conn = duckdb.connect('data/plant.duckdb', read_only=True)
print('Connected to plant.duckdb')
for t in plant_conn.execute('SHOW TABLES').fetchdf()['name']:
    count = plant_conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {count} rows')

Connected to plant.duckdb
  plant_companions: 409 rows
  plant_diseases: 219 rows
  plant_pests: 219 rows
  plant_types: 73 rows
  plant_varieties: 73 rows


In [25]:
# plant types
plant_conn.execute('SELECT * FROM plant_types ORDER BY category, name').fetchdf()

,plant_type_id,name,category
0,6bf1eb4a-7096-42e6-be0b-ee4078758f76,Basil,herbs
1,4c7f4a75-3141-4796-96ba-66f6213bc704,Bay Laurel,herbs
2,e2de69e9-0764-446e-b9ee-4c367bcadcaf,Borage,herbs
3,c89c3d95-8624-4a12-9fa7-00fd243ca760,Chervil,herbs
4,159f1922-0fff-418c-bdbb-3161afd78dbc,Chives,herbs
...,...,...,...
68,313bf9b9-92dd-45b9-97e9-3363b1955906,Tomatillos,vegetables
69,1e5fde50-9fe5-4ac2-b800-88834cdbd130,Tomatoes,vegetables
70,9e3e6229-aafa-4221-be1b-c167171aa1d1,Turnips,vegetables
71,509647c1-32c2-4f2a-bf46-431428da3b6a,Winter Squash,vegetables


In [26]:
plant_conn.execute("""
    SELECT outdoor_sow_date_range
    FROM plant_varieties
""").fetchdf()

,outdoor_sow_date_range
0,2-3 weeks before last frost
1,As soon as soil can be worked
2,4-6 weeks before last frost
3,Every two weeks for continuous harvest
4,2-4 weeks before last frost
...,...
68,After frost
69,After last frost
70,After the danger of frost has passed
71,After last spring frost


In [27]:
# varieties — growing data
plant_conn.execute("""
    SELECT pt.name, pt.category, pv.sun_tolerance, pv.water_required,
           pv.soil_n, pv.soil_p, pv.soil_k, pv.days_to_harvest,
           pv.spacing_inches, pv.temp_min_air_f, pv.temp_min_ground_f,
           pv.height_inches_estimate, pv.outdoor_sow_date_range,
           pv.indoor_sow_weeks_before_frost
    FROM plant_types pt
    JOIN plant_varieties pv ON pt.plant_type_id = pv.plant_type_id
    ORDER BY pt.category, pt.name
""").fetchdf()

,name,category,sun_tolerance,water_required,soil_n,soil_p,soil_k,days_to_harvest,spacing_inches,temp_min_air_f,temp_min_ground_f,height_inches_estimate,outdoor_sow_date_range,indoor_sow_weeks_before_frost
0,Basil,herbs,full_sun,medium,0.40,0.30,0.30,63,15.0,60.0,60.0,18.0,After last frost,6
1,Bay Laurel,herbs,partial_shade,low,0.30,0.20,0.30,730,42.0,40.0,40.0,120.0,n/a,8
2,Borage,herbs,partial_shade,medium,0.70,0.50,0.60,60,15.0,50.0,60.0,36.0,After the danger of frost has passed,6
3,Chervil,herbs,partial_shade,medium,0.30,0.30,0.20,50,9.0,45.0,45.0,18.0,Late spring,4
4,Chives,herbs,partial_shade,medium,0.80,0.50,0.50,60,10.0,45.0,40.0,18.0,March 15 - May 1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,Tomatillos,vegetables,full_sun,medium,0.60,0.60,0.65,85,30.0,60.0,55.0,42.0,Transplant when soil is 60°F,7
69,Tomatoes,vegetables,full_sun,high,0.85,0.70,0.80,80,21.0,60.0,60.0,48.0,Transplant when soil is 60°F,7
70,Turnips,vegetables,partial_shade,medium,0.60,0.50,0.50,30,5.0,45.0,40.0,15.0,March 15 - April 1,0
71,Winter Squash,vegetables,full_sun,high,0.85,0.75,0.80,95,36.0,70.0,70.0,18.0,When soil is at least 70°F,4


In [28]:
# companions and antagonists for a specific plant
PLANT = 'Tomatoes'  # change me

plant_conn.execute("""
    SELECT pc.companion_name, pc.relationship
    FROM plant_companions pc
    JOIN plant_types pt ON pc.plant_type_id = pt.plant_type_id
    WHERE LOWER(pt.name) = LOWER(?)
    ORDER BY pc.relationship, pc.companion_name
""", [PLANT]).fetchdf()

,companion_name,relationship
0,Corn,antagonist
1,kohlrabi,antagonist
2,potatoes,antagonist
3,Basil,companion
4,carrots,companion
5,onions,companion


In [29]:
# pests
plant_conn.execute("""
    SELECT pp.pest_name, pp.symptoms, pp.treatment
    FROM plant_pests pp
    JOIN plant_types pt ON pp.plant_type_id = pt.plant_type_id
    WHERE LOWER(pt.name) = LOWER(?)
""", [PLANT]).fetchdf()

,pest_name,symptoms,treatment
0,Tomato Hornworms,Chewed leaves,Hand-pick or use Bt spray
1,Aphids,Yellow leaves,Insecticidal soap or neem oil
2,Whiteflies,Sooty mold growth,Use yellow sticky traps


In [30]:
# diseases
plant_conn.execute("""
    SELECT pt.name, pd.disease_name, pd.symptoms, pd.treatment
    FROM plant_diseases pd
    JOIN plant_types pt ON pd.plant_type_id = pt.plant_type_id
    WHERE LOWER(pt.name) = LOWER(?)
""", [PLANT]).fetchdf()

,name,disease_name,symptoms,treatment
0,Tomatoes,Late Blight,Water-soaked lesions,"Remove affected plants, apply fungicides"
1,Tomatoes,Blossom End Rot,Black sunken spots,Ensure even watering and calcium
2,Tomatoes,Early Blight,Brown spots,Use copper fungicides and rotate crops


In [31]:
plant_conn.close()

---
## weather.duckdb

Populated by `build/weather_build/main.py` — 20yr daily history + frost dates + sow calendar.

In [32]:
weather_conn = duckdb.connect('data/weather.duckdb', read_only=True)
print('Connected to weather.duckdb')
for t in weather_conn.execute('SHOW TABLES').fetchdf()['name']:
    count = weather_conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {count} rows')

Connected to weather.duckdb
  frost_dates: 1 rows
  sow_calendar: 73 rows
  weather_metadata: 1 rows
  weather_records: 7427 rows


In [33]:
# metadata — zipcodes loaded and date range
weather_conn.execute('SELECT * FROM weather_metadata').fetchdf()

,zipcode,first_record_date,last_updated_date
0,98115,2006-01-01,2026-05-02


In [34]:
# frost dates per zipcode
weather_conn.execute('SELECT * FROM frost_dates').fetchdf()

,zipcode,avg_last_spring_frost,avg_first_fall_frost,frost_free_days,years_analyzed
0,98115,2026-03-06,2026-12-05,274,11


In [35]:
# sow calendar — indoor start + outdoor sow date per plant
ZIPCODE = '98115'  # change me

weather_conn.execute("""
    SELECT plant_name, indoor_start_date, outdoor_sow_date, notes
    FROM sow_calendar
    WHERE zipcode = ?
    ORDER BY outdoor_sow_date, plant_name
""", [ZIPCODE]).fetchdf()

,plant_name,indoor_start_date,outdoor_sow_date,notes
0,Arugula,NaT,2026-03-01,"Arugula can be directly sown in early spring, ..."
1,Bok Choy,2026-02-06,2026-03-01,Bok choy can be sown in early spring as it gro...
2,Carrots,NaT,2026-03-01,Carrots can be sown as soon as the soil can be...
3,Chervil,2026-02-06,2026-03-01,Chervil can be sown in early spring as it pref...
4,Cilantro,2026-02-13,2026-03-01,Cilantro prefers cooler temperatures and can b...
...,...,...,...,...
68,Tomatillos,2026-01-16,2026-05-15,Tomatillos should be planted outdoors after th...
69,Tomatoes,2026-01-16,2026-05-15,Tomatoes should be planted outdoors after the ...
70,Winter Squash,2026-02-06,2026-05-15,Winter squash should be planted after the last...
71,Zucchini,NaT,2026-05-15,Zucchini should be planted outdoors after the ...


In [36]:
# estimated outdoor sow dates — from two-agent DB range analysis (null = n/a range)
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT
        plant_name,
        estimated_outdoor_sow_date,
        CASE
            WHEN estimated_outdoor_sow_date IS NULL THEN 'n/a range'
            ELSE 'estimated'
        END AS status
    FROM sow_calendar
    WHERE zipcode = ?
    ORDER BY estimated_outdoor_sow_date NULLS LAST, plant_name
""", [ZIPCODE]).fetchdf()

,plant_name,estimated_outdoor_sow_date,status
0,Fava Beans,2026-01-25,estimated
1,Arugula,2026-02-05,estimated
2,Leeks,2026-02-06,estimated
3,Mustard Greens,2026-02-06,estimated
4,Asparagus,2026-02-13,estimated
...,...,...,...
68,Winter Squash,2026-06-01,estimated
69,Garlic,2026-10-29,estimated
70,Bay Laurel,NaT,n/a range
71,Lemongrass,NaT,n/a range


In [37]:
# sow calendar summary — counts by source and judge activity
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT
        sow_source,
        COUNT(*) AS plants,
        COUNT(estimated_outdoor_sow_date) AS has_estimate,
        COUNT(recommended_date) AS judge_resolved
    FROM sow_calendar
    WHERE zipcode = ?
    GROUP BY sow_source
    ORDER BY sow_source
""", [ZIPCODE]).fetchdf()

,sow_source,plants,has_estimate,judge_resolved
0,fallback,5,2,0
1,researched,68,68,49


In [38]:
# judge decisions — plants where final_judge_agent set a recommended_date
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT
        plant_name,
        outdoor_sow_date,
        estimated_outdoor_sow_date,
        recommended_date,
        recommended_source,
        ABS(DATEDIFF('day', outdoor_sow_date, estimated_outdoor_sow_date)) AS original_gap_days
    FROM sow_calendar
    WHERE zipcode = ?
      AND recommended_date IS NOT NULL
    ORDER BY original_gap_days DESC
""", [ZIPCODE]).fetchdf()

,plant_name,outdoor_sow_date,estimated_outdoor_sow_date,recommended_date,recommended_source,original_gap_days
0,Parsley,2026-03-15,2026-06-01,2026-03-15,researched,78
1,Chervil,2026-03-01,2026-05-06,2026-03-01,researched,66
2,Okra,2026-05-15,2026-03-13,2026-05-15,researched,63
3,Zucchini,2026-05-15,2026-03-13,2026-05-15,researched,63
4,Melons,2026-05-15,2026-03-13,2026-05-15,researched,63
5,Corn,2026-05-15,2026-03-13,2026-05-15,researched,63
6,Stevia,2026-05-15,2026-03-13,2026-05-15,researched,63
7,Basil,2026-05-15,2026-03-13,2026-05-15,researched,63
8,Rosemary,2026-05-15,2026-03-13,2026-05-15,researched,63
9,Sage,2026-05-15,2026-03-13,2026-05-15,researched,63


In [39]:
# disagreements — plants where researched and estimated dates differ by more than 14 days
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT
        plant_name,
        sow_source,
        outdoor_sow_date,
        estimated_outdoor_sow_date,
        ABS(DATEDIFF('day', outdoor_sow_date, estimated_outdoor_sow_date)) AS gap_days
    FROM sow_calendar
    WHERE zipcode = ?
      AND outdoor_sow_date IS NOT NULL
      AND estimated_outdoor_sow_date IS NOT NULL
      AND ABS(DATEDIFF('day', outdoor_sow_date, estimated_outdoor_sow_date)) > 14
    ORDER BY gap_days DESC
""", [ZIPCODE]).fetchdf()

,plant_name,sow_source,outdoor_sow_date,estimated_outdoor_sow_date,gap_days
0,Parsley,researched,2026-03-15,2026-06-01,78
1,Chervil,researched,2026-03-01,2026-05-06,66
2,Okra,researched,2026-05-15,2026-03-13,63
3,Zucchini,researched,2026-05-15,2026-03-13,63
4,Melons,researched,2026-05-15,2026-03-13,63
5,Corn,researched,2026-05-15,2026-03-13,63
6,Stevia,researched,2026-05-15,2026-03-13,63
7,Basil,researched,2026-05-15,2026-03-13,63
8,Rosemary,researched,2026-05-15,2026-03-13,63
9,Sage,researched,2026-05-15,2026-03-13,63


In [40]:
# sow calendar — all date columns: researched, estimated, and judge's recommendation
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT
        plant_name,
        indoor_start_date,
        outdoor_sow_date,
        estimated_outdoor_sow_date,
        recommended_date,
        recommended_source,
        sow_source
    FROM sow_calendar
    WHERE zipcode = ?
    ORDER BY COALESCE(recommended_date, outdoor_sow_date), plant_name
""", [ZIPCODE]).fetchdf()

,plant_name,indoor_start_date,outdoor_sow_date,estimated_outdoor_sow_date,recommended_date,recommended_source,sow_source
0,Arugula,NaT,2026-03-01,2026-02-05,2026-03-01,researched,researched
1,Bok Choy,2026-02-06,2026-03-01,2026-02-20,NaT,None,researched
2,Carrots,NaT,2026-03-01,2026-03-15,NaT,None,researched
3,Chervil,2026-02-06,2026-03-01,2026-05-06,2026-03-01,researched,researched
4,Cilantro,2026-02-13,2026-03-01,2026-03-13,NaT,None,researched
...,...,...,...,...,...,...,...
68,Tomatillos,2026-01-16,2026-05-15,2026-06-01,2026-05-15,researched,researched
69,Tomatoes,2026-01-16,2026-05-15,2026-06-01,2026-05-15,researched,researched
70,Winter Squash,2026-02-06,2026-05-15,2026-06-01,2026-05-15,researched,researched
71,Zucchini,NaT,2026-05-15,2026-03-13,2026-05-15,researched,researched


In [41]:
# recent weather — last 30 days
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT date, temp_min_f, temp_max_f, precipitation_in, wind_max_mph, wind_direction
    FROM weather_records
    WHERE zipcode = ?
    ORDER BY date DESC
    LIMIT 30
""", [ZIPCODE]).fetchdf()

,date,temp_min_f,temp_max_f,precipitation_in,wind_max_mph,wind_direction
0,2026-05-02,50.0,69.2,0.004,12.7,4°
1,2026-05-01,49.7,69.3,0.000,8.7,10°
2,2026-04-30,47.4,70.5,0.000,7.8,359°
3,2026-04-29,46.8,62.0,0.000,9.8,4°
4,2026-04-28,48.7,57.2,0.035,9.3,192°
5,2026-04-27,47.1,58.4,0.020,11.1,194°
6,2026-04-26,42.8,65.3,0.000,6.3,237°
7,2026-04-25,41.9,62.0,0.000,8.3,3°
8,2026-04-24,44.2,60.3,0.000,11.2,360°
9,2026-04-23,47.1,57.6,0.000,10.5,207°


In [42]:
# monthly averages — temp and precipitation by month
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT
        MONTH(date) AS month,
        ROUND(AVG(temp_min_f), 1) AS avg_low_f,
        ROUND(AVG(temp_max_f), 1) AS avg_high_f,
        ROUND(SUM(precipitation_in) / COUNT(DISTINCT YEAR(date)), 2) AS avg_annual_precip_in
    FROM weather_records
    WHERE zipcode = ?
    GROUP BY MONTH(date)
    ORDER BY month
""", [ZIPCODE]).fetchdf()

,month,avg_low_f,avg_high_f,avg_annual_precip_in
0,1,36.3,45.4,7.02
1,2,35.9,46.5,4.51
2,3,38.2,50.7,5.32
3,4,41.5,55.6,3.82
4,5,47.4,62.6,2.56
5,6,52.6,67.8,2.30
6,7,57.2,75.1,0.78
7,8,57.8,75.0,1.21
8,9,53.9,68.5,2.67
9,10,46.9,58.2,5.11


In [43]:
# coldest days on record
ZIPCODE = '98115'

weather_conn.execute("""
    SELECT date, temp_min_f, temp_max_f, precipitation_in
    FROM weather_records
    WHERE zipcode = ?
    ORDER BY temp_min_f ASC
    LIMIT 10
""", [ZIPCODE]).fetchdf()

,date,temp_min_f,temp_max_f,precipitation_in
0,2008-12-20,13.0,23.3,0.260
1,2024-01-13,13.8,25.6,0.000
2,2006-11-29,14.8,34.9,0.079
3,2008-12-19,15.7,25.0,0.000
4,2010-11-23,15.7,25.4,0.000
5,2010-11-24,16.6,28.6,0.000
6,2011-02-25,17.5,30.2,0.000
7,2011-02-26,17.7,33.5,0.051
8,2022-01-01,17.8,35.1,0.000
9,2024-01-12,18.0,28.8,0.000


In [44]:
weather_conn.close()